# PSPFL — Federated, Privacy-Preserving Version of the KNN Fatigue Model

This notebook builds a **Profile-Similarity Personalized Federated Learning (PSPFL)**
model for fatigue classification and compares it against the centralized KNN from
`knn_task_loto.ipynb`. It follows the structure of the reference notebooks
(`HAR_PSPFL_LSTM`, `HAR_PSPFL_CNN`): local training per client → profile-similarity
personalization → federated aggregation.

### Why this design

* **Same inputs as the KNN.** We reuse the *top-2 input recipes* surfaced by the KNN
  Stage-1 sweep (sensor type × statistic × anthropometrics). Set them in the config cell.
* **A weight-averageable base learner.** KNN has no trainable parameters, so it cannot be
  federated by weight averaging. PSPFL/FedAvg average model *weights* across clients, so we
  swap in a **linear model with shareable weights** — `SGDClassifier` trained with
  `partial_fit` (true local epochs). We run it in two flavours: **linear SVM** (`hinge`
  loss) and **logistic regression** (`log_loss`). Neither is RF/XGBoost, and both consume
  the exact same feature vectors the KNN used.
* **Decentralized & privacy-preserving.** Each subject is a *client* that keeps its own raw
  windows. Only model weights (and a scalar sample count) are exchanged. The personalization
  signal uses only coarse anthropometrics (gender, age, BMI, waist-hip ratio) via a cosine
  similarity matrix — never raw sensor data.

### What we compare (all evaluated on the same per-client held-out test sets)

| Model | Data location | Personalized? |
|-------|---------------|---------------|
| **Centralized KNN** | all raw data pooled on a server | no |
| **FedAvg** (SVM / Logistic) | data stays on-device | no (one shared global model) |
| **PSPFL** (SVM / Logistic) | data stays on-device | yes (similarity-blended per client) |

The story: PSPFL recovers (and ideally beats) the centralized KNN **without ever pooling
raw data**, while adding per-worker personalization.


## 0. Imports & configuration

Set `TOP2_RECIPES` to the two best recipes from your KNN Stage-1 table (same
`sensors / stats / anthro` vocabulary as `knn_task_loto.ipynb`). Defaults below are
plausible strong recipes — **replace them with your actual top-2.**


In [ ]:
import pickle, warnings, itertools, copy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import SGDClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
from sklearn.model_selection import GridSearchCV, GroupKFold

warnings.filterwarnings("ignore")
SEED = 1365
np.random.seed(SEED)

# ── Paths (reuse the KNN feature cache if it exists) ─────────────────────────
DATA_PATH  = "NIOSH_Combined_Dataset.pkl"
OUTPUT_DIR = Path("outputs"); OUTPUT_DIR.mkdir(exist_ok=True)
CACHE_CSV  = OUTPUT_DIR / "task_features.csv"   # written by knn_task_loto.ipynb

# ── Cohort / tasks / sensors (must match the KNN notebook) ───────────────────
EXCLUDE_SUBJECTS = {"Sub11", "Sub12", "Sub14"}
TASKS = [{"Weight": 2.5, "Pace": 15}, {"Weight": 2.5, "Pace": 10},
         {"Weight": 2.5, "Pace": 5},  {"Weight": 1.5, "Pace": 15}]
TASK_NAMES = [f"T{i} (W{t['Weight']}/P{t['Pace']})" for i, t in enumerate(TASKS)]

BODY_SEGMENTS = ["trunk", "upper_arm", "wrist"]
SENSOR_TYPES  = ["Accelerometer", "Gyroscope"]          # magnetometer excluded
AXES          = ["X", "Y", "Z"]
SENSOR_COLS   = [f"{seg}_{st}.{ax}" for seg in BODY_SEGMENTS
                 for st in SENSOR_TYPES for ax in AXES]   # 18 channels
STATS         = ["mean", "std", "rms"]
ANTHRO_COLS   = ["anthro_Gender", "anthro_Age", "anthro_BMI", "anthro_WHR"]

FS, WINDOW_SIZE, STRIDE = 100, 1000, 500
MVIC_MINUTES, MVIC_BUFFER_SECONDS = [9, 18, 27, 36, 45], 60
RPE_BIN_EDGES, RPE_LABELS = [-0.01, 3.5, 6.5, 10.01], ["Low", "Moderate", "High"]
CLASSES = np.array([0, 1, 2])      # global label set shared by every client

# ── >>> TOP-2 KNN INPUT RECIPES — EDIT THESE <<< ─────────────────────────────
# Each recipe: which sensor types, which per-channel stats, include anthro in the
# *feature vector*? (Anthro is always used for the similarity matrix regardless.)
TOP2_RECIPES = [
    {"name": "acc+gyro|all|+anthro",
     "sensors": ("Accelerometer", "Gyroscope"), "stats": ("mean", "std", "rms"), "anthro": True},
    {"name": "acc+gyro|mean|+anthro",
     "sensors": ("Accelerometer", "Gyroscope"), "stats": ("mean",), "anthro": True},
]

# ── Federated hyperparameters (kept modest; sweep where useful) ──────────────
COMM_ROUNDS   = 12          # server<->client communication rounds
LOCAL_EPOCHS  = 10          # partial_fit passes per client per round
RATE          = 1.0         # global blend scale (matches reference 'rate')
RATIO_GRID    = [0.0, 0.1, 0.3, 0.5]   # PSPFL personalization strength to sweep
SGD_LOSSES    = {"svm": "hinge", "logistic": "log_loss"}  # two federated learners
SGD_ALPHA     = 1e-4        # L2 regularisation strength
NORMALIZE_BLEND = True      # divide similarity blend by sum of sims (numerical stability)
TEST_FRAC     = 0.30        # per-client hold-out fraction

print("Federated config ready.")
print("Top-2 recipes:", [r["name"] for r in TOP2_RECIPES])


## 1. Feature matrix (reuse KNN cache, else rebuild)

If `outputs/task_features.csv` exists (produced by the KNN notebook) we load it. Otherwise
we rebuild the exact same windowed `mean/std/rms` features from the pickle so this notebook
stands alone.


In [ ]:
def session_to_task(settings, subject, session, tasks=TASKS):
    """Resolve a (subject, session) to its task index by matching (Weight, Pace)."""
    if subject not in settings:
        return None
    df_set = pd.DataFrame(settings[subject]).copy()
    df_set.columns = [str(c).capitalize() for c in df_set.columns]
    if session not in df_set.index:
        return None
    w, p = float(df_set.loc[session, "Weight"]), float(df_set.loc[session, "Pace"])
    for ti, t in enumerate(tasks):
        if np.isclose(w, t["Weight"]) and np.isclose(p, t["Pace"]):
            return ti
    return None


def build_anthro_lookup(anthro):
    """{subject -> anthropometric feature dict}; used both as features and for similarity."""
    a = anthro.copy()
    a["anthro_Gender"] = a["Gender"].astype(str).str.upper().str[0].map({"M": 1, "F": 0})
    a["anthro_Age"]    = pd.to_numeric(a["Age"], errors="coerce")
    a["anthro_BMI"]    = a["Weight (kg)"] / (a["Height (cm)"] / 100.0) ** 2
    a["anthro_WHR"]    = a["Waist circumference (cm)"] / a["Hip circumference (cm)"]
    return a.set_index("Subject")[ANTHRO_COLS].to_dict("index")


def _bin_rpe(v):
    out = pd.cut([v], bins=RPE_BIN_EDGES, labels=[0, 1, 2])[0]
    return int(out) if not pd.isna(out) else (0 if v <= 3.5 else (1 if v <= 6.5 else 2))


def rebuild_features(data_path):
    """Rebuild the windowed feature matrix from the pickle (mirrors the KNN notebook)."""
    with open(data_path, "rb") as f:
        data = pickle.load(f)
    ts_data  = {k: v for k, v in data["ts_data"].items() if k[0] not in EXCLUDE_SUBJECTS}
    anthro   = data["anthro_clean"]
    anthro   = anthro[~anthro["Subject"].isin(EXCLUDE_SUBJECTS)].reset_index(drop=True)
    settings = data["experiment_settings"]
    lkp      = build_anthro_lookup(anthro)

    rows = []
    for (subj, sess), df in sorted(ts_data.items()):
        task = session_to_task(settings, subj, sess)
        if task is None:
            continue
        if "Session" in df.columns:
            df = df[df["Session"].astype(str).str.strip() == sess].copy()
        if df.empty or any(c not in df.columns for c in SENSOR_COLS):
            continue
        rpe = df["RPE_Val"].astype(float).interpolate("linear").bfill().ffill().values
        imu = df[SENSOR_COLS].values.astype(np.float32)
        elapsed = df["Timestamp"].values - df["Timestamp"].iloc[0]
        mvic = np.zeros(len(df), bool)
        for tm in MVIC_MINUTES:
            mvic |= np.abs(elapsed - tm * 60) <= MVIC_BUFFER_SECONDS
        n, start = len(df), 0
        while start + WINDOW_SIZE <= n:
            sl = slice(start, start + WINDOW_SIZE)
            if mvic[sl].any() or np.isnan(imu[sl]).any():
                start += STRIDE; continue
            rr = rpe[sl][~np.isnan(rpe[sl])]
            if rr.size == 0:
                start += STRIDE; continue
            win = imu[sl]
            row = {"Subject": subj, "Task": task, "y": _bin_rpe(float(np.median(rr)))}
            mean, std = win.mean(0), win.std(0)
            rms = np.sqrt((win ** 2).mean(0))
            for j, ch in enumerate(SENSOR_COLS):
                row[f"{ch}_mean"], row[f"{ch}_std"], row[f"{ch}_rms"] = \
                    float(mean[j]), float(std[j]), float(rms[j])
            row.update(lkp.get(subj, {}))
            rows.append(row); start += STRIDE
    return pd.DataFrame(rows)


if Path(CACHE_CSV).exists():
    print(f"Loading cached features from {CACHE_CSV}")
    feat_df = pd.read_csv(CACHE_CSV)
else:
    print("Cache not found — rebuilding features from pickle ...")
    feat_df = rebuild_features(DATA_PATH)

SENSOR_FEATS = [f"{ch}_{s}" for ch in SENSOR_COLS for s in STATS]
feat_df = feat_df.dropna(subset=SENSOR_FEATS + ANTHRO_COLS + ["y"]).reset_index(drop=True)
print(f"{len(feat_df):,} windows from {feat_df['Subject'].nunique()} subjects (clients)")
feat_df["y"].map(dict(enumerate(RPE_LABELS))).value_counts()


## 2. Recipe → feature columns

`select_columns()` maps a recipe to its feature-column list, exactly as in the KNN notebook.


In [ ]:
def select_columns(sensors, stats, anthro):
    """Columns for one recipe: sensor-type x statistic, optionally + anthropometrics."""
    cols = [f"{ch}_{st}" for ch in SENSOR_COLS for st in stats
            if any(s in ch for s in sensors)]
    return cols + ANTHRO_COLS if anthro else cols

for r in TOP2_RECIPES:
    r["cols"] = select_columns(r["sensors"], r["stats"], r["anthro"])
    print(f"{r['name']:24s} -> {len(r['cols'])} features")


## 3. Clients & global feature scaling

Each **subject is a client**. We split every client's windows into a local train/test set
(stratified by fatigue class where possible). A single `StandardScaler` is fit on the pooled
training windows and broadcast to all clients — this mirrors the reference's *pre-standardized*
data and keeps weight-averaging coherent (in deployment these are shared normalization
constants, not raw data).


In [ ]:
def per_client_split(y, test_frac=TEST_FRAC, seed=SEED):
    """Stratified per-client train/test indices; falls back gracefully for tiny classes."""
    rng = np.random.default_rng(seed)
    tr, te = [], []
    for c in np.unique(y):
        ci = np.where(y == c)[0]; rng.shuffle(ci)
        n_tr = max(1, int(round(len(ci) * (1 - test_frac))))
        n_tr = min(n_tr, len(ci) - 1) if len(ci) > 1 else len(ci)
        tr += list(ci[:n_tr]); te += list(ci[n_tr:])
    return np.array(tr, int), np.array(te, int)


def build_clients(feat_df, cols):
    """
    Partition feat_df into per-client arrays and fit a global scaler on pooled train data.

    Returns
    -------
    clients : list of dicts {subject, Xtr, ytr, Xte, yte}  (scaled features)
    order   : list of subject ids (defines the client index used by the similarity matrix)
    """
    order = sorted(feat_df["Subject"].unique())
    raw = {}
    pooled_Xtr = []
    for s in order:
        d = feat_df[feat_df["Subject"] == s]
        X = d[cols].values.astype(float); y = d["y"].values.astype(int)
        tr, te = per_client_split(y)
        raw[s] = (X[tr], y[tr], X[te], y[te])
        pooled_Xtr.append(X[tr])

    scaler = StandardScaler().fit(np.vstack(pooled_Xtr))  # global normalization constants
    clients = []
    for s in order:
        Xtr, ytr, Xte, yte = raw[s]
        clients.append({"subject": s,
                        "Xtr": scaler.transform(Xtr), "ytr": ytr,
                        "Xte": scaler.transform(Xte), "yte": yte})
    return clients, order, scaler

# sanity check on the first recipe
_clients, _order, _ = build_clients(feat_df, TOP2_RECIPES[0]["cols"])
print(f"{len(_clients)} clients; example train/test sizes: "
      f"{[ (c['Xtr'].shape[0], c['Xte'].shape[0]) for c in _clients[:3] ]}")


## 4. Profile-similarity matrix

Cosine similarity between min-max-normalized anthropometric profiles (gender, age, BMI,
waist-hip ratio) — the only cross-client information PSPFL uses for personalization.


In [ ]:
def build_similarity(feat_df, order):
    """Cosine similarity (clientCount x clientCount) over normalized anthropometrics."""
    prof = (feat_df.groupby("Subject")[ANTHRO_COLS].first().loc[order]).values.astype(float)
    mn, mx = prof.min(0), prof.max(0)
    prof_norm = (prof - mn) / (mx - mn + 1e-12)
    return cosine_similarity(prof_norm)

cos_sim = build_similarity(feat_df, _order)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cos_sim, cmap="YlOrRd", vmin=0, vmax=1)
ax.set_xticks(range(len(_order))); ax.set_yticks(range(len(_order)))
ax.set_xticklabels(_order, rotation=90, fontsize=7); ax.set_yticklabels(_order, fontsize=7)
plt.colorbar(im, ax=ax, label="cosine similarity")
ax.set_title("Participant profile similarity\n(gender, age, BMI, WHR)", fontsize=11)
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "pspfl_similarity.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. Federated learner internals (shareable linear weights)

An `SGDClassifier`'s parameters are `coef_` (n_classes × n_features) and `intercept_`
(n_classes) — directly averageable and blendable, just like the dense layers in the
reference LSTM/CNN. These helpers get/set those weights, run a warm-started local update,
and predict from a raw weight set.


In [ ]:
def make_sgd(loss):
    """Fresh linear model: 'hinge' = linear SVM, 'log_loss' = logistic regression."""
    return SGDClassifier(loss=loss, alpha=SGD_ALPHA, learning_rate="optimal",
                         random_state=SEED)

def get_w(clf):
    """Extract [coef_, intercept_] as a copyable weight list."""
    return [clf.coef_.copy(), clf.intercept_.copy()]

def set_w(clf, W):
    """Inject a [coef_, intercept_] weight list into a (shape-initialized) classifier."""
    clf.coef_ = W[0].copy(); clf.intercept_ = W[1].copy()

def zeros_w(n_features):
    """Server-side zero initialization with the correct multiclass shapes."""
    k = len(CLASSES)
    return [np.zeros((k, n_features)), np.zeros(k)]

def local_update(Xtr, ytr, init_W, loss, local_epochs=LOCAL_EPOCHS):
    """
    One client's local training: warm-start from init_W (server or own weights),
    then run `local_epochs` shuffled partial_fit passes. Raw data never leaves here.
    Returns the updated weight list.
    """
    clf = make_sgd(loss)
    clf.partial_fit(Xtr[:min(len(Xtr), 3)], ytr[:min(len(ytr), 3)], classes=CLASSES)  # init shapes
    if init_W is not None:
        set_w(clf, init_W)
    rng = np.random.default_rng(SEED)
    for _ in range(local_epochs):
        idx = rng.permutation(len(Xtr))
        clf.partial_fit(Xtr[idx], ytr[idx])
    return get_w(clf)

def predict_w(W, X, loss="hinge"):
    """Predict labels from a raw weight set (shape-init a classifier, inject W)."""
    clf = make_sgd(loss)
    clf.partial_fit(X[:min(len(X), 3)], np.array([CLASSES[i % len(CLASSES)]
                    for i in range(min(len(X), 3))]), classes=CLASSES)
    set_w(clf, W)
    return clf.predict(X)

def eval_W(W, X, y, loss="hinge"):
    """Macro-F1 / accuracy / precision / recall for a weight set on (X, y)."""
    if len(y) == 0:
        return dict(macro_f1=np.nan, accuracy=np.nan, precision=np.nan, recall=np.nan)
    p = predict_w(W, X, loss)
    return dict(macro_f1=f1_score(y, p, average="macro", zero_division=0),
                accuracy=accuracy_score(y, p),
                precision=precision_score(y, p, average="macro", zero_division=0),
                recall=recall_score(y, p, average="macro", zero_division=0))


## 6. FedAvg and PSPFL training loops

**FedAvg** — every round each client warm-starts from the shared server model, trains
locally, and the server takes a sample-size-weighted average. One global model is evaluated
on every client's test set (no personalization).

**PSPFL** — every client keeps a *persistent personalized* weight set. Each round: train
locally, then blend in other clients' fresh weights scaled by profile similarity
(`W_i += ratio·rate·Σ_j sim_ij·W_j`, optionally similarity-normalized for stability). Each
client is evaluated with its *own* personalized model. `ratio = 0` reduces PSPFL to
independent local models.


In [ ]:
def run_fedavg(clients, loss, rounds=COMM_ROUNDS):
    """Vanilla federated averaging. Returns (per-client metrics list, test-F1 history)."""
    n_feat = clients[0]["Xtr"].shape[1]
    W_server = zeros_w(n_feat)
    sizes = np.array([len(c["ytr"]) for c in clients], float)
    hist = []
    for _ in range(rounds):
        locals_ = [local_update(c["Xtr"], c["ytr"], W_server, loss) for c in clients]
        # sample-weighted average of client weights -> new global model
        W_server = [sum(sizes[i] / sizes.sum() * locals_[i][l] for i in range(len(clients)))
                    for l in range(len(W_server))]
        f1s = [eval_W(W_server, c["Xte"], c["yte"], loss)["macro_f1"] for c in clients]
        hist.append(np.nanmean(f1s))
    metrics = [eval_W(W_server, c["Xte"], c["yte"], loss) for c in clients]
    return metrics, hist


def run_pspfl(clients, loss, sim, ratio, rounds=COMM_ROUNDS,
              rate=RATE, normalize=NORMALIZE_BLEND):
    """
    Profile-similarity personalized FL. Each client keeps its own weights across rounds;
    similarity-weighted contributions from other clients are blended in each round.
    Returns (per-client metrics list, mean test-F1 history).
    """
    n = len(clients); n_feat = clients[0]["Xtr"].shape[1]
    W = [zeros_w(n_feat) for _ in range(n)]            # persistent per-client weights
    hist = []
    for _ in range(rounds):
        fresh = [local_update(clients[i]["Xtr"], clients[i]["ytr"], W[i], loss)
                 for i in range(n)]                    # local training from own weights
        new_W = []
        for i in range(n):
            denom = sum(sim[i][j] for j in range(n) if j != i) if normalize else 1.0
            denom = denom if denom > 1e-12 else 1.0
            blended = [layer.copy() for layer in fresh[i]]
            for j in range(n):
                if j == i:
                    continue
                coeff = ratio * rate * sim[i][j] / denom
                for l in range(len(blended)):
                    blended[l] += fresh[j][l] * coeff
            new_W.append(blended)
        W = new_W
        f1s = [eval_W(W[i], clients[i]["Xte"], clients[i]["yte"], loss)["macro_f1"]
               for i in range(n)]
        hist.append(np.nanmean(f1s))
    metrics = [eval_W(W[i], clients[i]["Xte"], clients[i]["yte"], loss) for i in range(n)]
    return metrics, hist


## 7. Centralized KNN baseline (the non-private reference)

The original model: pool every client's training windows on one server, fit a KNN, and
evaluate on each client's test set. `k` is tuned with subject-grouped CV. This is exactly
what PSPFL avoids — here it is the privacy-invasive yardstick.


In [ ]:
def run_centralized_knn(clients, k_grid=(5, 11, 21)):
    """Pool all client train data, tune k (subject-grouped), eval per client. Returns metrics."""
    Xtr = np.vstack([c["Xtr"] for c in clients])
    ytr = np.concatenate([c["ytr"] for c in clients])
    groups = np.concatenate([[c["subject"]] * len(c["ytr"]) for c in clients])
    gkf = GroupKFold(n_splits=min(5, len(clients)))
    search = GridSearchCV(KNeighborsClassifier(weights="distance", n_jobs=-1),
                          {"n_neighbors": list(k_grid)}, scoring="f1_macro",
                          cv=gkf, n_jobs=-1).fit(Xtr, ytr, groups=groups)
    knn = search.best_estimator_
    metrics = []
    for c in clients:
        if len(c["yte"]) == 0:
            continue
        p = knn.predict(c["Xte"])
        metrics.append(dict(macro_f1=f1_score(c["yte"], p, average="macro", zero_division=0),
                            accuracy=accuracy_score(c["yte"], p),
                            precision=precision_score(c["yte"], p, average="macro", zero_division=0),
                            recall=recall_score(c["yte"], p, average="macro", zero_division=0)))
    return metrics, search.best_params_["n_neighbors"]


## 8. Run the full comparison

For each of the top-2 recipes: centralized KNN, then FedAvg and PSPFL (over the ratio grid)
for both the SVM and logistic learners. We record the **mean across clients** of macro-F1 and
accuracy, plus convergence histories.


In [ ]:
def mean_metric(metrics, key):
    return float(np.nanmean([m[key] for m in metrics]))

results, histories = [], {}
for recipe in TOP2_RECIPES:
    clients, order, _ = build_clients(feat_df, recipe["cols"])
    sim = build_similarity(feat_df, order)

    # --- centralized KNN baseline ---
    knn_m, best_k = run_centralized_knn(clients)
    results.append(dict(recipe=recipe["name"], model="KNN (centralized)", learner="knn",
                        ratio=np.nan, k=best_k,
                        macro_f1=mean_metric(knn_m, "macro_f1"),
                        accuracy=mean_metric(knn_m, "accuracy")))
    print(f"[{recipe['name']}] KNN(centralized) k={best_k}: "
          f"macroF1={results[-1]['macro_f1']:.3f}")

    # --- federated learners ---
    for lname, loss in SGD_LOSSES.items():
        fa_m, fa_hist = run_fedavg(clients, loss)
        results.append(dict(recipe=recipe["name"], model=f"FedAvg ({lname})", learner=lname,
                            ratio=np.nan, k=np.nan,
                            macro_f1=mean_metric(fa_m, "macro_f1"),
                            accuracy=mean_metric(fa_m, "accuracy")))
        histories[(recipe["name"], f"FedAvg ({lname})")] = fa_hist
        print(f"[{recipe['name']}] FedAvg({lname}): macroF1={results[-1]['macro_f1']:.3f}")

        for ratio in RATIO_GRID:
            ps_m, ps_hist = run_pspfl(clients, loss, sim, ratio)
            results.append(dict(recipe=recipe["name"], model=f"PSPFL ({lname}) r={ratio}",
                                learner=lname, ratio=ratio, k=np.nan,
                                macro_f1=mean_metric(ps_m, "macro_f1"),
                                accuracy=mean_metric(ps_m, "accuracy")))
            histories[(recipe["name"], f"PSPFL ({lname}) r={ratio}")] = ps_hist
            print(f"[{recipe['name']}] PSPFL({lname}) ratio={ratio}: "
                  f"macroF1={results[-1]['macro_f1']:.3f}")

res_df = pd.DataFrame(results).sort_values(["recipe", "macro_f1"], ascending=[True, False])
res_df.reset_index(drop=True, inplace=True)
res_df


## 9. Comparison plots

Best model per family (KNN vs FedAvg vs PSPFL) for each recipe, and PSPFL convergence vs
the FedAvg baseline.


In [ ]:
# (a) best-per-family bar chart
def family(m):
    if m.startswith("KNN"): return "KNN (centralized)"
    if m.startswith("FedAvg"): return "FedAvg"
    return "PSPFL"

res_df["family"] = res_df["model"].map(family)
best = (res_df.groupby(["recipe", "family"])["macro_f1"].max().unstack("family"))
order_fam = [f for f in ["KNN (centralized)", "FedAvg", "PSPFL"] if f in best.columns]
best = best[order_fam]

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(best.index)); w = 0.25
colors = {"KNN (centralized)": "#8D99AE", "FedAvg": "#457B9D", "PSPFL": "#E63946"}
for i, fam in enumerate(order_fam):
    ax.bar(x + (i - 1) * w, best[fam].values, w, label=fam, color=colors[fam])
ax.set_xticks(x); ax.set_xticklabels(best.index, rotation=10, ha="right")
ax.set_ylabel("mean per-client macro-F1"); ax.set_ylim(0, 1)
ax.set_title("Centralized KNN vs Federated FedAvg vs PSPFL", fontweight="bold")
ax.legend(); ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "pspfl_family_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
best.round(4)


In [ ]:
# (b) convergence: best PSPFL vs FedAvg for recipe 0 (SVM learner)
rname = TOP2_RECIPES[0]["name"]
fig, ax = plt.subplots(figsize=(9, 5))
for key, h in histories.items():
    if key[0] != rname:
        continue
    if "svm" not in key[1] and "FedAvg (svm)" != key[1]:
        continue
    style = "--" if key[1].startswith("FedAvg") else "-"
    ax.plot(range(1, len(h) + 1), h, style, marker="o", markersize=3, label=key[1])
ax.set_xlabel("communication round"); ax.set_ylabel("mean per-client macro-F1")
ax.set_title(f"Federated convergence — {rname} (SVM learner)", fontweight="bold")
ax.legend(fontsize=8); ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "pspfl_convergence.png", dpi=150, bbox_inches="tight")
plt.show()


## 10. Privacy / decentralization notes & conclusions

**What stayed private.** Raw IMU windows never left any client; only model weights and a
scalar training-set size were communicated. The personalization channel used four coarse
anthropometric numbers per subject (not raw signals). FedAvg and PSPFL therefore satisfy the
"data stays on device" property the centralized KNN violates.

**Optional hardening (mentioned, not run):** add Gaussian noise to client weight updates for
(ε, δ)-differential privacy, or secure-aggregation so the server only ever sees the summed
update. Both slot into `run_fedavg` / `run_pspfl` at the weight-exchange step.

The summary cell reports, per recipe, how the best federated model compares to the
centralized KNN and whether profile-similarity personalization (PSPFL) beat plain FedAvg.


In [ ]:
res_df.to_csv(OUTPUT_DIR / "pspfl_vs_knn_results.csv", index=False)

print("=" * 72)
print("SUMMARY — federated (PSPFL/FedAvg) vs centralized KNN")
print("=" * 72)
for rname in [r["name"] for r in TOP2_RECIPES]:
    sub = res_df[res_df["recipe"] == rname]
    knn  = sub[sub["family"] == "KNN (centralized)"]["macro_f1"].max()
    fa   = sub[sub["family"] == "FedAvg"]["macro_f1"].max()
    ps_row = sub[sub["family"] == "PSPFL"].sort_values("macro_f1", ascending=False).iloc[0]
    ps   = ps_row["macro_f1"]
    print(f"\nRecipe: {rname}")
    print(f"  centralized KNN macro-F1 : {knn:.3f}  (pools all raw data)")
    print(f"  best FedAvg     macro-F1 : {fa:.3f}  (decentralized, no personalization)")
    print(f"  best PSPFL      macro-F1 : {ps:.3f}  (decentralized, ratio={ps_row['ratio']})")
    print(f"    PSPFL - FedAvg = {ps - fa:+.3f}   (personalization gain)")
    print(f"    PSPFL - KNN    = {ps - knn:+.3f}   (federated vs centralized)")
print("\nSaved: pspfl_vs_knn_results.csv, pspfl_similarity.png, "
      "pspfl_family_comparison.png, pspfl_convergence.png  (in outputs/)")
print("=" * 72)
